In [2]:
# =========================
# Task 14 - ETL Mini Pipeline
# =========================

import pandas as pd
import sqlite3
import os

# Step 1: Create folders for organization
os.makedirs("raw", exist_ok=True)
os.makedirs("processed", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Step 2: Load your single CSV file (use full path if needed)
# Example: r"C:\Users\HP\Downloads\Transaction.csv"
df = pd.read_csv(r"C:\Users\HP\Downloads\retail_sales_dataset.csv")

# Save a raw copy inside the raw/ folder
df.to_csv("raw/retail_raw.csv", index=False)

# Step 3: Clean missing values + duplicates
df = df.dropna().drop_duplicates()

# Step 4: Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Step 5: Create derived columns
# Example margin calculation (assuming cost = 80% of unit price)
df["margin"] = df["total_amount"] - (df["quantity"] * df["price_per_unit"] * 0.8)

# Segment flag by age
df["segment_flag"] = df["age"].apply(
    lambda x: "young" if x < 30 else "senior" if x > 50 else "adult"
)

# Step 6: Split into logical tables
customers = df[["customer_id", "gender", "age"]].drop_duplicates()
orders = df[["transaction_id", "customer_id", "date", "total_amount"]]
products = df[["transaction_id", "product_category", "quantity", "price_per_unit"]]

# Step 7: Save processed outputs
df.to_csv("processed/processed_data.csv", index=False)
customers.to_csv("processed/customers.csv", index=False)
orders.to_csv("processed/orders.csv", index=False)
products.to_csv("processed/products.csv", index=False)

# Step 8: Load into SQLite
conn = sqlite3.connect("output/database.sqlite")
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
conn.close()

# Step 9: Validate counts
print("Raw count:", len(pd.read_csv("raw/retail_raw.csv")))
print("Processed count:", len(df))
print("Customers:", len(customers))
print("Orders:", len(orders))
print("Products:", len(products))

print("ETL pipeline completed successfully!")


Raw count: 1000
Processed count: 1000
Customers: 1000
Orders: 1000
Products: 1000
ETL pipeline completed successfully!


In [ ]:
import pandas as pd
import sqlite3
import os

# Load your dataset
df = pd.read_csv(r"C:\Users\HP\Downloads\retail_sales_dataset.csv")

# Example cleaning
df = df.dropna().drop_duplicates()
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Create split tables
customers = df[["customer_id", "gender", "age"]].drop_duplicates()
orders = df[["transaction_id", "customer_id", "date", "total_amount"]]
products = df[["transaction_id", "product_category", "quantity", "price_per_unit"]]
# Export processed dataset
df.to_csv("processed/processed_data.csv", index=False)

# Export split tables
customers.to_csv("processed/customers.csv", index=False)
orders.to_csv("processed/orders.csv", index=False)
products.to_csv("processed/products.csv", index=False)

print("CSV files exported successfully!")
